# 03. Feature Engineering

ベースライン(約 0.9412)から **+0.003〜0.004** を積んだ工程。効いたものと効かなかったものを
実際に動かして確認する。関数の中身は `03_fe.ipynb` を参照。

## 結論を先に

| 施策 | 効果 | 理由 |
|---|---|---|
| **厳密値 Target Encoding** | **+0.003 前後** | 最大の改善要因 |
| Count Encoding | +0.0005〜0.0008 | TE と非冗長で加算的(CatBoost では無効) |
| Triple TE + digit + ビン1024 | +0.0005〜0.001 | 上位カーネル由来 |
| 四則演算 (diff/ratio/sum/avg) | **無効〜悪化** | 生成過程に交互作用がない |
| 交互作用 TE(2〜13列) | **すべて無効** | 同上 |

In [ ]:
import os, sys
# リポジトリルートを作業ディレクトリにして、data/ などの相対パスを揃える
if os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")
sys.path.insert(0, os.path.abspath("src"))
print("cwd:", os.getcwd())

In [ ]:
import numpy as np
import pandas as pd
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score
from lightgbm import LGBMClassifier

TARGET = "Will_Buy_EV"
train = pd.read_csv("data/train.csv")
test = pd.read_csv("data/test.csv")
y = (train[TARGET] == "Yes").astype(int)
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

NUM_COLS = [c for c in train.select_dtypes(include=[np.number]).columns if c != "id"]
CAT_COLS = [c for c in train.columns if c not in NUM_COLS + ["id", TARGET]]
ALL_COLS = NUM_COLS + CAT_COLS

## なぜ Target Encoding が効くのか

EDA で見たとおり年収は 13,214 種類の値を持つ。木は既定で 255 ビンにまとめるため、
**値ごとに違う購入率を直接は学べない**。TE はその「値ごとの購入率」を 1 列で渡す。

数値列もビン分割せず **厳密な値のままキーにする**のが要点。

In [ ]:
rate = y.groupby(train["Annual_Income_USD"]).agg(["mean", "size"])
print("年収のユニーク値:", len(rate))
print("1値あたりの平均行数:", round(rate["size"].mean(), 1))
rate.head()

## リーク対策 — 入れ子 CV

TE は目的変数を使うため、作り方を誤ると学習データの答えが漏れる。ここでは二重に防ぐ。

1. 外側 fold の学習データ内だけで統計を計算する
2. **学習行には内側 CV の out-of-fold 値**を当てる(自分のラベルを含まない値にする)

2 は単なる保険ではなく、XGBoost では **+0.00108** の精度向上につながった。
学習時だけ TE が当たりすぎる状態(楽観バイアス)が解消されるため。

In [ ]:
def target_encode(tr_key, tr_y, va_key, prior, smooth=20.0, inner_splits=5):
    """外側foldの学習データで fit し、学習行には内側OOF値を当てる入れ子TE。"""
    # valid/test 用: 学習fold全体の統計
    agg = tr_y.groupby(tr_key).agg(["sum", "count"])
    m = (agg["sum"] + prior * smooth) / (agg["count"] + smooth)
    va_te = va_key.map(m).fillna(prior).astype("float32")

    # 学習行用: 内側CVのOOF値
    tr_te = pd.Series(np.full(len(tr_key), prior, dtype="float32"), index=tr_key.index)
    inner = StratifiedKFold(n_splits=inner_splits, shuffle=True, random_state=42)
    for i_tr, i_va in inner.split(tr_key, tr_y):
        a = tr_y.iloc[i_tr].groupby(tr_key.iloc[i_tr]).agg(["sum", "count"])
        mm = (a["sum"] + prior * smooth) / (a["count"] + smooth)
        tr_te.iloc[i_va] = tr_key.iloc[i_va].map(mm).fillna(prior).astype("float32").values
    return tr_te, va_te

## 評価用の CV ループ

特徴量の作り方(`build`)を差し替えて A/B するための関数。
`build` は fold ごとに呼ばれ、学習用と検証用の特徴量を返す。

In [ ]:
def evaluate(build, n_splits=5, n_estimators=300, seed=42):
    oof = np.zeros(len(train))
    for fold, (tr, va) in enumerate(skf.split(train, y)):
        Xtr, Xva = build(tr, va)
        model = LGBMClassifier(n_estimators=n_estimators, learning_rate=0.1,
                               random_state=seed, verbosity=-1)
        model.fit(Xtr, y.iloc[tr])
        oof[va] = model.predict_proba(Xva)[:, 1]
        if fold + 1 >= n_splits:
            break
    mask = np.zeros(len(train), dtype=bool)
    for i, (_, va) in enumerate(skf.split(train, y)):
        mask[va] = True
        if i + 1 >= n_splits:
            break
    score = roc_auc_score(y[mask], oof[mask])
    print(f"AUC: {score:.5f}")
    return score

## A. ベースライン(生の特徴量のみ)

時間短縮のため 2 fold・300 本で比較する。数値は本番(5 fold・収束設定)とは揃わないが、
施策同士の比較には使える。

In [ ]:
def prep_cat(df):
    out = df.copy()
    for c in CAT_COLS:
        cats = pd.concat([train[c], test[c]]).astype("category").cat.categories
        out[c] = pd.Categorical(out[c], categories=cats)
    return out

X_raw = prep_cat(train[ALL_COLS])

def build_base(tr, va):
    return X_raw.iloc[tr], X_raw.iloc[va]

score_base = evaluate(build_base, n_splits=2)

## B. + 厳密値 Target Encoding(全13列)

In [ ]:
prior = y.mean()

def build_te(tr, va):
    Xtr, Xva = X_raw.iloc[tr].copy(), X_raw.iloc[va].copy()
    for c in ALL_COLS:
        key = train[c].astype(str)
        t, v = target_encode(key.iloc[tr], y.iloc[tr], key.iloc[va], prior)
        Xtr[f"te_{c}"], Xva[f"te_{c}"] = t.values, v.values
    return Xtr, Xva

score_te = evaluate(build_te, n_splits=2)
print(f"ベースラインとの差: {score_te - score_base:+.5f}")

## C. + Count Encoding

値の出現頻度。**目的変数を使わないので train+test をまとめて数えてよい**(リークしない)。
TE とは別の情報なので加算的に効く。

In [ ]:
count_maps = {c: pd.concat([train[c], test[c]]).astype(str).value_counts() for c in ALL_COLS}

def build_te_cnt(tr, va):
    Xtr, Xva = build_te(tr, va)
    for c in ALL_COLS:
        key = train[c].astype(str)
        Xtr[f"cnt_{c}"] = key.iloc[tr].map(count_maps[c]).values
        Xva[f"cnt_{c}"] = key.iloc[va].map(count_maps[c]).values
    return Xtr, Xva

score_te_cnt = evaluate(build_te_cnt, n_splits=2)
print(f"TE のみとの差: {score_te_cnt - score_te:+.5f}")

## D. + 四則演算(効かない例)

「充電スタンド数の自宅+職場」「年収÷通勤距離」など直感的な合成指標。
木は 1 列ずつしか分割できないので効きそうに見えるが、**合成データの生成過程に
そうした関係がない**ため効かない。3モデルすべてで無効〜悪化だった。

In [ ]:
PAIRS = [("Charging_Stations_Near_Home", "Charging_Stations_Near_Work"),
         ("Annual_Income_USD", "Daily_Commute_km"),
         ("Age", "Annual_Income_USD")]

def build_arith(tr, va):
    Xtr, Xva = build_te_cnt(tr, va)
    for a, b in PAIRS:
        for X_, idx in ((Xtr, tr), (Xva, va)):
            X_[f"{a}_sum_{b}"] = train[a].iloc[idx].values + train[b].iloc[idx].values
            X_[f"{a}_diff_{b}"] = train[a].iloc[idx].values - train[b].iloc[idx].values
            X_[f"{a}_ratio_{b}"] = train[a].iloc[idx].values / (train[b].iloc[idx].values + 1e-6)
    return Xtr, Xva

score_arith = evaluate(build_arith, n_splits=2)
print(f"TE+Count との差: {score_arith - score_te_cnt:+.5f}")

## 結果のまとめ

In [ ]:
pd.DataFrame({
    "AUC": [score_base, score_te, score_te_cnt, score_arith],
    "前からの差": [np.nan, score_te - score_base, score_te_cnt - score_te, score_arith - score_te_cnt],
}, index=["A. 生の特徴量", "B. + 厳密値TE", "C. + Count Encoding", "D. + 四則演算"]).round(5)

## 本番の構成

ここまでの結果に、上位カーネル由来の施策を足したものが本番。

- **Triple TE**: smooth(縮約の強さ)を変えた TE を 3 本**同時に**入れる
- **Smooth Keys**: 年収を /10、/100、/1000 に丸めたものも TE のキーにする
- **digit features**: 数値を桁ごとにばらして列にする
- **ビン数 1024**: 既定の 255 では年収の細かい違いが潰れるため

本番の実行は `src/<model>_preprocessing.py`(コマンドは README 参照)。

| モデル | OOF AUC | Baseline | Diff | Rank |
|---|---|---|---|---| 
| XGBoost | 0.94591 | | | 1 | 
| CatBoost | 0.94589 | | | 2 |
| RealMLP | 0.94589 | # | | 2 | 
| LightGBM | 0.94587 |  | | 4 | 
| **4モデルアンサンブル** | **0.94618** | | # | # | 